# VA DEQ Site Join

The purpose of this notebook is to combine the old and the new embeddedness datasets provided by the Virginia Department of Environmental Quality. The new dataset (ProbMon) contains many pre-calculated attributes for each site. The old dataset (TMDLsummary) contains attributes needed to calculate shear stress. The final dataset will be exported to Excel then analyzed for correlations in a separate Jnotebook.

E. Reilly Oare

In [1]:
# Standard Libraries
import os  # File handling and directory management

# Data Handling
import pandas as pd  # Data manipulation and analysis
import numpy as np # Number handling

## Step 1. Import datasets

In [2]:
# Define base directory
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "../.."))

In [3]:
# Read in old excel file
old_emd_path = os.path.join(BASE_DIR, "data", "raw", "virginia", "TMDLsummary_2023-06-01.xlsx")
oldemd = pd.read_excel(old_emd_path, header = 0)

# Read in new excel file (has coordinates attached to site)
new_emd_path = os.path.join(BASE_DIR, "data", "raw", "virginia", "Probmon2001-2022.xlsx")
newemd = pd.read_excel(new_emd_path, header = 0)

# Read in coordinates
coords_path = os.path.join(BASE_DIR, "data", "GIS", "vadeq_sites_albers.xlsx")
coords = pd.read_excel(coords_path)

## Step 2. Check for null values

In [4]:
# Since our target feature is embeddedness, we will search for null values in that column only
print("The number of null values in the old dataset is", oldemd['Xembed'].isnull().sum().sum())
print("The number of null values in the new dataset is", newemd['Embed_PCT'].isnull().sum().sum())

The number of null values in the old dataset is 0
The number of null values in the new dataset is 243


*For now, we will need to delete the 243 datapoints that do not have embeddedness since that is our target.*

In [5]:
# Delete all data points in new dataset (ProbMon) that don't have embeddedness measurements
newemd.dropna(subset = 'Embed_PCT', inplace = True)

print("The number of null values in the new dataset is now", newemd['Embed_PCT'].isnull().sum().sum())

The number of null values in the new dataset is now 0


## Step 3. Rename datasets for joining

In [6]:
# Rename columns in newemd to match oldemd
newemd = newemd.rename(columns={'Embed_PCT':'Xembed',
                      'BL_CB_GR_Embed_PCT':'BL_CB_GR_transectPCT'
                      })

In [7]:
# Create Year column in old dataset for indexing
oldemd['Year'] = oldemd['Date'].dt.year

## Step 4. Drop uneccessary columns

In [8]:
# Tables were opened in Excel to identify columns of interest

# Drop columns in oldemd
oldemd.drop(columns=['SampleID',
                     'Interval',
                     'SiteFlag',
                     'ThalwegN'], inplace = True)

# Drop columns in newemd
newemd.drop(columns=['DataSource',
                     'StationID_Trend',
                     'stratum',
                     'designweight',
                     'weightcategory',
                     'station',
                     'state',
                     'status',
                     'comment',
                     'set',
                     'Order',
                     'BasinSize',
                     'StreamSizeCatPhase',
                     'IR2008',
                     'IR2010',
                     'IR2012',
                     'IR2014',
                     'IR2016',
                     'IR2018',
                     'IR2020',
                     'IR2022',
                     'IR2024',
                     'Hg-C',
                     'YearSampled',
                     'NLCD',
                     'MunMajor',
                     'MunMinor',
                     'IndMajor',
                     'IndMinor',
                    ], 
            inplace = True)

## Step 5. Join Tables

In [9]:
# Use a left join
emd = oldemd.merge(newemd, 
                   on = ['StationID',
                       'Year',
                       'Xembed'],
                   how = "left")

In [10]:
# Use another left join
vadeq_emd = emd.merge(coords,
                      on = 'StationID',
                      how = 'left')

In [11]:
# Display joined dataframe
vadeq_emd

,StationID,Date,ReachLength,Slope_x,RP100,BR_PCT,HP_PCT,RC_PCT,BL_PCT,CB_PCT,...,POPCHG2010_2020,POPCHG2000_2020,RDLEN,RDLEN120,STXRD_CNT,RDDENS,pctRoadLengthInRiparian,STXRD,LATITUDE,LONGITUDE
0,1AACO006.10,2006-11-21,440,0.220,34.752500,7.692308,6.730769,0.000000,9.615385,14.423077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.728611,-77.203333
1,1AACO004.84,2008-06-25,320,0.521,25.757012,0.000000,0.952381,0.000000,5.714286,27.619048,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.720500,-77.190722
2,1AACO006.10,2008-06-26,520,0.173,35.429907,2.857143,7.619048,0.952381,4.761905,21.904762,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.728611,-77.203333
3,1AACO009.14,2008-06-26,560,0.223,22.451871,2.857143,4.761905,0.000000,9.523810,18.095238,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.761944,-77.207222
4,1AAUA017.60,2005-09-22,160,0.400,19.910507,0.000000,0.000000,0.000000,7.619048,34.285714,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.490361,-77.466389
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1120,2BXRK001.64,2022-10-26,150,1.287,9.486085,0.952381,5.714286,0.952381,0.000000,13.333333,...,-3.340934,-19.035539,5757.591719,1574.982250,2.0,2.716352,27.354879,0.894433,37.811383,-78.729900
1121,2-JKS070.06,2022-09-12,800,0.450,25.065800,9.523810,0.000000,0.000000,5.714286,30.476190,...,-8.892221,-3.953434,451582.048000,86994.700400,69.0,4.955175,19.264428,0.960441,38.123444,-79.779725
1122,4ASNA007.82,2022-09-20,320,0.430,15.910317,0.000000,0.000000,0.000000,0.000000,0.000000,...,-6.086899,-1.849916,617340.395400,53983.910780,90.0,1.004842,8.744594,0.451753,36.792139,-79.112444
1123,8-MIC001.47,2022-10-25,150,0.300,10.014865,0.000000,9.708738,0.000000,0.000000,0.000000,...,5.915693,-2.301971,58550.497110,7208.478259,12.0,1.883880,12.311558,0.815454,37.753450,-77.720931


In [ ]:
# Push to processed data folder for use in EDA, etc.
unclean_dir = os.path.join(BASE_DIR, "data", "processed", "virginia", "uncleaned_vadeq_emd_joined.xlsx")
vadeq_emd.to_excel(unclean_dir)

## Step 6. Clean up VA DEQ dataset

In [ ]:
# Check dataset for null values
percent_missing = vadeq_emd.isnull().sum()*100/len(vadeq_emd) # Calculates percent missing
missing_value_df = pd.DataFrame({'column_name':vadeq_emd.columns,
                                'percent_missing':percent_missing})
missing_value_df.sort_values('percent_missing', inplace=True)

# Display dataframe
missing_value_df.sort_values(by = 'percent_missing',
                            ascending = False)

In [ ]:
# Create a list of columns that have >80% of values missing
missing_cols = []
for index, row in missing_value_df.iterrows():
    if row['percent_missing'] > 80:
        missing_cols.append(row['column_name'])

missing_cols    

In [ ]:
# Remove columns from vdeq_emd that are missing more than 80% of their data
for i in missing_cols:
    if i in vadeq_emd.columns:
        vadeq_emd.drop(columns=i, inplace=True)

# Display trimmed dataframe        
vadeq_emd

## Step 7. Add physics-informed attribute

We're going to add bankfull shear velocity by using the bankfull width and bankfull width to depth ratio. The equation for $u^*$ is:
$$u^* = \sqrt{g * R_h * S}$$
Where $g$ is gravity,
<br> $R_h$ is hydraulic radius (assuming rectangular channel), and
<br> $S$ is slope.

In [ ]:
# Define variables
g = 9.81 # m/s^2
W_bkfl = vadeq_emd['XBKF_W']
H_bkfl = vadeq_emd['BKF_depth_in_meters']
S = vadeq_emd['Slope_x'] / 100

# Do calculations
A = W_bkfl * H_bkfl
P = W_bkfl + (2 * H_bkfl)
R_h = A / P
vadeq_emd['Bankfull Shear Velocity (m/s)'] = np.sqrt(g * R_h * S)

# Display number of null valus
vadeq_emd[['Bankfull Shear Velocity (m/s)']].isnull().sum()

## Step 8. Export Trimmed Dataset

In [ ]:
# Push trimmed vadeq dataset to processed data folder
trimmed_path = os.path.join(BASE_DIR, "data", "processed", "virginia", "trimmed_vadeq_emd.xlsx")
vadeq_emd.to_excel(trimmed_path)
print(f"Dataset saved to {trimmed_path}")